# 08 - Consolidación del dataset final

Junta las 7 fuentes ya analizadas (incluye VinDr-CXR, subconjunto de Tuberculosis, agregado tras el cruce de hashes en 07_cruce_hashes.ipynb) (EDA + revisión manual de calidad + cruce de hashes entre fuentes) en **un solo dataset final**, guardado como:
- Una copia física única de las imágenes seleccionadas, organizadas por clase (`../dataset_final/images/<clase>/...`) — RSNA convertido de DICOM a PNG, el resto se resguarda también como PNG por consistencia (la resolución original NO se toca, solo cambia el contenedor de archivo).
- Un **manifiesto maestro** en CSV con una fila por imagen (ruta final, clase, split, fuente, y las banderas para filtrar variantes) y, a partir de él, **4 manifiestos de variante** — no 4 copias del dataset:
  1. 6 clases, sin excluir candidatas de calidad
  2. 6 clases, excluyendo candidatas de calidad
  3. 5 clases (sin STB), sin excluir candidatas de calidad
  4. 5 clases (sin STB), excluyendo candidatas de calidad

Decisiones ya confirmadas: techo de 6,000 para Normal/Pneumonia/COVID-19 (Tuberculosis/Silicosis/STB se usan al 100%, no llegan al techo); split 70/15/15 decidido **una sola vez** sobre el pool ya capado (las 4 variantes son subconjuntos de ese mismo split, no splits independientes); split agrupado por paciente para SilicoData (ahí se había encontrado fuga de paciente); STB se descarta por completo en la variante de 5 clases.

## 0. Configuración

In [ ]:
import random
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from tqdm import tqdm

sys.path.insert(0, ".")
import eda_utils as eu

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

HASH_DIR = Path("../Datasets/_eda_hashes")
EDA_DIR = Path("../resultados/eda")
FINAL_DIR = Path("../dataset_final")
IMAGES_DIR = FINAL_DIR / "images"
MANIFESTS_DIR = FINAL_DIR / "manifiestos"
SUMMARY_DIR = FINAL_DIR / "resumen"
for d in (IMAGES_DIR, MANIFESTS_DIR, SUMMARY_DIR):
    d.mkdir(parents=True, exist_ok=True)

ABUNDANT_CLASSES = {"Normal", "Pneumonia", "COVID-19"}
TARGET_CEILING = 6000  # solo aplica a ABUNDANT_CLASSES; Tuberculosis/Silicosis/STB usan el 100%
SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}

## 1. Cargar el pool combinado y aplicar las exclusiones ya decididas

Tres exclusiones se aplican aquí, no antes:
1. **Duplicados cruzados entre fuentes** (`duplicados_cruzados_a_descartar.csv`, del notebook 06) — nos quedamos con la copia de mejor resolución de cada grupo.
2. **`Non-COVID` de COVID-QU-Ex** — nunca se cuenta como Pneumonia (decisión de diseño del estudio). Como COVID-QU-Ex es la única fuente que aportó `Pneumonia` fuera de Chest_x_ray/RSNA, basta con excluir todas las filas `source_dataset == "COVID-QU-Ex" & target_class == "Pneumonia"`.
3. Set B de SilicoData no se carga (no se generó un hash CSV para él a propósito — ver `05_eda_silicodata.ipynb`, se dejó como recurso de anotación, no como fuente de imágenes).

In [ ]:
hash_files = sorted(HASH_DIR.glob("*_md5.csv"))
pool = pd.concat([pd.read_csv(f) for f in hash_files], ignore_index=True)
print(f"Pool bruto (6 fuentes, sin ninguna exclusión todavía): {len(pool)}")

descartar_cruzados = pd.read_csv(HASH_DIR / "duplicados_cruzados_a_descartar.csv")
pool = pool[~pool["path"].isin(set(descartar_cruzados["path"]))].copy()
print(f"Tras quitar duplicados cruzados entre fuentes: {len(pool)}")

es_non_covid_covidqu = (pool["source_dataset"] == "COVID-QU-Ex") & (pool["target_class"] == "Pneumonia")
print(f"Filas de Non-COVID/COVID-QU-Ex excluidas de Pneumonia: {es_non_covid_covidqu.sum()}")
pool = pool[~es_non_covid_covidqu].copy()
print(f"Pool neto tras las 2 exclusiones: {len(pool)}")

In [ ]:
print("Pool neto por clase y fuente:")
pool.groupby(["target_class", "source_dataset"]).size().unstack(fill_value=0)

## 2. Aplicar las decisiones de calidad (revisión manual)
Se leen `decisiones_calidad.csv` y `exclusiones_manuales_adicionales.csv` de las 6 carpetas de `resultados/eda/` — de ambos solo interesan `path` y `decision`; el resto de columnas varía entre fuentes y no hace falta para esto.

In [ ]:
review_frames = []
for f in EDA_DIR.glob("*/decisiones_calidad.csv"):
    review_frames.append(pd.read_csv(f)[["path", "decision"]])
for f in EDA_DIR.glob("*/exclusiones_manuales_adicionales.csv"):
    df = pd.read_csv(f)
    if len(df):
        review_frames.append(df[["path", "decision"]])

reviews = pd.concat(review_frames, ignore_index=True) if review_frames else pd.DataFrame(columns=["path", "decision"])
excluir_calidad = set(reviews.loc[reviews["decision"] == "excluir", "path"])
print(f"Total de imágenes marcadas para excluir por calidad (algoritmo corregido a mano + agregadas a mano): {len(excluir_calidad)}")

pool["calidad_excluir"] = pool["path"].isin(excluir_calidad)
print("Por clase:")
pool.groupby("target_class")["calidad_excluir"].sum()

**Nota**: las rutas de `exclusiones_manuales_adicionales.csv` que no correspondan a ninguna fila del pool (por ejemplo, si apuntan a una imagen de Set B de SilicoData, que no está en este pool) simplemente no producen efecto — no rompen nada, solo no encuentran coincidencia en `isin`.

## 3. Resumen por fuente (antes de aplicar el techo de 6,000)
Cuánto aportó cada dataset a cada clase, ya con las exclusiones de las secciones 1 y 2 aplicadas.

In [ ]:
contribucion = pool.groupby(["target_class", "source_dataset"]).size().unstack(fill_value=0)
contribucion = contribucion.reindex(
    ["Normal", "Pneumonia", "COVID-19", "Tuberculosis", "Silicosis", "STB"]
)
contribucion.to_csv(SUMMARY_DIR / "contribucion_por_fuente.csv")

ax = contribucion.plot(kind="bar", stacked=True, figsize=(9, 5), colormap="tab10")
ax.set_title("Pool disponible por clase, desglosado por fuente (antes del techo de 6,000)")
ax.set_ylabel("Número de imágenes")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(SUMMARY_DIR / "contribucion_por_fuente.png", dpi=120)
plt.show()

## 4. Aplicar el techo de 6,000 a Normal/Pneumonia/COVID-19

Para no dejar que una sola fuente grande (ej. COVID-QU-Ex) se coma todo el cupo y borre la diversidad de origen que se buscó desde el principio, el techo se reparte con un algoritmo simple de *water-filling*: se reparte el cupo en partes iguales entre las fuentes disponibles, y si alguna fuente no tiene suficientes imágenes para su parte, el sobrante se redistribuye entre las demás.

Tuberculosis, Silicosis y STB no pasan por esto — se usa el 100% de lo disponible en las tres, porque ya están muy por debajo del techo.

In [ ]:
def water_fill(target_total: int, capacities: dict) -> dict:
    alloc = {s: 0 for s in capacities}
    remaining_target = min(target_total, sum(capacities.values()))
    active = set(capacities)
    while remaining_target > 0 and active:
        share = remaining_target / len(active)
        given_this_round = 0
        for s in list(active):
            give = min(share, capacities[s] - alloc[s])
            alloc[s] += give
            given_this_round += give
            if alloc[s] >= capacities[s] - 1e-9:
                active.discard(s)
        remaining_target -= given_this_round
        if given_this_round == 0:
            break
    return {s: int(round(v)) for s, v in alloc.items()}

In [ ]:
selected_parts = []
cap_report = {}

for target_class, group in pool.groupby("target_class"):
    if target_class not in ABUNDANT_CLASSES:
        selected_parts.append(group)
        cap_report[target_class] = {"disponible": len(group), "seleccionado": len(group)}
        continue
    capacities = group.groupby("source_dataset").size().to_dict()
    alloc = water_fill(TARGET_CEILING, capacities)
    parts = []
    for source, n in alloc.items():
        if n == 0:
            continue
        sub = group[group["source_dataset"] == source]
        parts.append(sub.sample(n=n, random_state=RANDOM_SEED))
    selected = pd.concat(parts, ignore_index=True) if parts else group.iloc[0:0]
    selected_parts.append(selected)
    cap_report[target_class] = {"disponible": len(group), "seleccionado": len(selected)}

pool_capado = pd.concat(selected_parts, ignore_index=True)
pd.DataFrame(cap_report).T

## 5. Split train/val/test (70/15/15), agrupado por paciente donde se conoce

Una sola vez, sobre `pool_capado`. Para las filas de SilicoData (única fuente con `patient_id` conocido) se agrupa por paciente con `GroupShuffleSplit`, para que ningún paciente quede repartido entre dos splits — exactamente lo que faltaba en el split oficial de SilicoData. Para el resto (sin ID de paciente disponible) se hace un split aleatorio estratificado por fuente, para que cada split mantenga la mezcla de orígenes poblacionales, no solo el azar.

In [ ]:
def split_group(df_class: pd.DataFrame) -> pd.Series:
    """Devuelve una Serie 'train'/'val'/'test' alineada al índice de df_class."""
    result = pd.Series(index=df_class.index, dtype=object)

    with_patient = df_class[df_class["patient_id"].notna()] if "patient_id" in df_class else df_class.iloc[0:0]
    without_patient = df_class.drop(with_patient.index)

    if len(with_patient):
        gss1 = GroupShuffleSplit(n_splits=1, train_size=SPLIT_RATIOS["train"], random_state=RANDOM_SEED)
        train_idx, rest_idx = next(gss1.split(with_patient, groups=with_patient["patient_id"]))
        rest = with_patient.iloc[rest_idx]
        val_frac_of_rest = SPLIT_RATIOS["val"] / (SPLIT_RATIOS["val"] + SPLIT_RATIOS["test"])
        gss2 = GroupShuffleSplit(n_splits=1, train_size=val_frac_of_rest, random_state=RANDOM_SEED)
        val_idx, test_idx = next(gss2.split(rest, groups=rest["patient_id"]))
        result.loc[with_patient.iloc[train_idx].index] = "train"
        result.loc[rest.iloc[val_idx].index] = "val"
        result.loc[rest.iloc[test_idx].index] = "test"

    if len(without_patient):
        strat = without_patient["source_dataset"] if without_patient["source_dataset"].nunique() > 1 else None
        try:
            train_p, rest_p = train_test_split(
                without_patient, train_size=SPLIT_RATIOS["train"], random_state=RANDOM_SEED, stratify=strat
            )
            strat_rest = rest_p["source_dataset"] if strat is not None else None
            val_frac_of_rest = SPLIT_RATIOS["val"] / (SPLIT_RATIOS["val"] + SPLIT_RATIOS["test"])
            val_p, test_p = train_test_split(
                rest_p, train_size=val_frac_of_rest, random_state=RANDOM_SEED, stratify=strat_rest
            )
        except ValueError:
            # alguna fuente con muy pocas imágenes para estratificar -> split simple sin estratificar
            train_p, rest_p = train_test_split(without_patient, train_size=SPLIT_RATIOS["train"], random_state=RANDOM_SEED)
            val_frac_of_rest = SPLIT_RATIOS["val"] / (SPLIT_RATIOS["val"] + SPLIT_RATIOS["test"])
            val_p, test_p = train_test_split(rest_p, train_size=val_frac_of_rest, random_state=RANDOM_SEED)
        result.loc[train_p.index] = "train"
        result.loc[val_p.index] = "val"
        result.loc[test_p.index] = "test"

    return result

pool_capado["split_final"] = pd.Series(index=pool_capado.index, dtype=object)
for target_class, group in pool_capado.groupby("target_class"):
    pool_capado.loc[group.index, "split_final"] = split_group(group)

print("Split final por clase:")
pool_capado.groupby(["target_class", "split_final"]).size().unstack(fill_value=0)

In [ ]:
# Verificación correcta: lo que importa es que un mismo paciente, DENTRO DE LA MISMA CLASE, no
# quede repartido entre splits (memorizar al paciente sí ayudaría a "adivinar" ahí). Si el mismo
# paciente aparece en más de una clase (posible en un dataset longitudinal: sano en una visita,
# enfermo en otra), eso no es una fuga -- es información clínica real, y se reporta aparte.
silico = pool_capado[pool_capado["patient_id"].notna()] if "patient_id" in pool_capado else pool_capado.iloc[0:0]

fuga_por_clase = silico.groupby(["target_class", "patient_id"])["split_final"].nunique()
print(f"Pacientes repartidos entre splits DENTRO DE LA MISMA CLASE (esto sí sería fuga real, debe ser 0): {(fuga_por_clase > 1).sum()}")

clases_por_paciente = silico.groupby("patient_id")["target_class"].nunique()
pacientes_multi_clase = clases_por_paciente[clases_por_paciente > 1]
print(f"Pacientes con imágenes en más de una clase (dataset longitudinal, no es fuga): {len(pacientes_multi_clase)}")
if len(pacientes_multi_clase):
    display(silico[silico["patient_id"].isin(pacientes_multi_clase.index)].groupby(
        ["patient_id", "target_class", "split_final"]
    ).size())

## 6. Copiar y convertir las imágenes al árbol final

Se copian **todas** las imágenes de `pool_capado` (la variante más amplia: 6 clases, sin excluir candidatas de calidad) — las otras 3 variantes son manifiestos que apuntan a este mismo árbol de archivos, no copias adicionales. RSNA se convierte de DICOM a PNG; el resto se re-guarda como PNG también por consistencia, sin cambiar la resolución.

Nombre de archivo final: `<fuente>__<nombre_original_saneado>.png` — evita choques de nombre entre fuentes y mantiene trazabilidad al origen.

In [ ]:
import re

def safe_stem(path: str) -> str:
    stem = Path(path).stem
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", stem)

def save_as_png(src_path: str, dst_path: Path) -> bool:
    try:
        if src_path.endswith((".dcm", ".dicom")):
            arr, err = eu.probe_dicom_normalized(src_path)
            if err is not None:
                return False
            Image.fromarray(arr).save(dst_path)
        else:
            with Image.open(src_path) as im:
                im.save(dst_path)
        return True
    except Exception as e:
        print(f"Error con {src_path}: {e}")
        return False

final_paths = []
used_names = set()
for row in tqdm(pool_capado.itertuples(), total=len(pool_capado), desc="Copiando/convirtiendo imágenes"):
    class_dir = IMAGES_DIR / row.target_class
    class_dir.mkdir(parents=True, exist_ok=True)
    base_name = f"{row.source_dataset}__{safe_stem(row.path)}.png"
    name = base_name
    n = 1
    while name in used_names:
        name = f"{row.source_dataset}__{safe_stem(row.path)}_{n}.png"
        n += 1
    used_names.add(name)
    dst_path = class_dir / name
    ok = save_as_png(row.path, dst_path)
    final_paths.append(str(dst_path) if ok else None)

pool_capado["path_final"] = final_paths
fallidas = pool_capado["path_final"].isna().sum()
print(f"Imágenes que fallaron al copiar/convertir: {fallidas}")
pool_capado = pool_capado.dropna(subset=["path_final"]).copy()

## 7. Manifiesto maestro y las 4 variantes

In [ ]:
manifiesto = pool_capado[[
    "path_final", "target_class", "split_final", "source_dataset", "calidad_excluir", "phash", "md5", "patient_id"
]].rename(columns={"split_final": "split"})
manifiesto["es_stb"] = manifiesto["target_class"] == "STB"
manifiesto.to_csv(MANIFESTS_DIR / "manifiesto_maestro.csv", index=False)
print(f"Manifiesto maestro: {len(manifiesto)} imágenes -> {MANIFESTS_DIR / 'manifiesto_maestro.csv'}")

In [ ]:
variantes = {
    "6clases_sin_excluir": manifiesto,
    "6clases_con_exclusion": manifiesto[~manifiesto["calidad_excluir"]],
    "5clases_sin_excluir": manifiesto[~manifiesto["es_stb"]],
    "5clases_con_exclusion": manifiesto[(~manifiesto["es_stb"]) & (~manifiesto["calidad_excluir"])],
}

for nombre, df_variant in variantes.items():
    out_path = MANIFESTS_DIR / f"manifiesto_{nombre}.csv"
    df_variant.drop(columns=["calidad_excluir", "es_stb"]).to_csv(out_path, index=False)
    print(f"{nombre}: {len(df_variant)} imágenes -> {out_path}")

## 8. Resumen final por clase y por variante

In [ ]:
resumen_variantes = pd.DataFrame({
    nombre: df_variant.groupby("target_class").size() for nombre, df_variant in variantes.items()
}).fillna(0).astype(int)
resumen_variantes.to_csv(SUMMARY_DIR / "resumen_variantes.csv")
resumen_variantes

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
resumen_variantes.plot(kind="bar", ax=ax)
ax.set_title("Cantidad final por clase, en cada una de las 4 variantes")
ax.set_ylabel("Número de imágenes")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(SUMMARY_DIR / "resumen_variantes.png", dpi=120)
plt.show()

In [ ]:
print("Split final (train/val/test) de la variante más amplia (6 clases, sin excluir):")
manifiesto.groupby(["target_class", "split"]).size().unstack(fill_value=0)

In [ ]:
import subprocess
size_bytes = sum(f.stat().st_size for f in IMAGES_DIR.rglob("*") if f.is_file())
print(f"Peso total de dataset_final/images: {size_bytes / (1024**3):.2f} GB")

## Próximos pasos

1. Revisar el peso total de arriba — si es demasiado para subir cómodo a Drive, la única fuente de verdad que se puede ajustar sin rehacer nada es `TARGET_CEILING` o redimensionar en un paso extra (la resolución original se conservó a propósito, por decisión ya tomada).
2. Comprimir `dataset_final/` (imágenes + manifiestos) y subirlo una sola vez a Google Drive.
3. En Colab: montar Drive, leer el manifiesto de la variante que se quiera entrenar ese día (`manifiesto_6clases_con_exclusion.csv`, etc.), y cargar las imágenes por la columna `path_final` más el `split` para armar train/val/test.
4. Actualizar `class_weights.json` con los conteos finales de la variante que se entrene primero.